# 课程设计

- 学号：3250102780
- 姓名：李昊泽


In [1]:
import copy
import math
import os
import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

SEED = 20260910
TARGET_CLASSES = (0, 3, 6)
CLASS_NAMES = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
default_mode = 'kaggle' if device.type == 'cuda' else 'local'
RUN_MODE = os.getenv('CIFAR10_RUN_MODE', default_mode).strip().lower()

PROFILES = {
    'smoke': dict(depth=16, width=1, dropout=0.1, epochs=1, batch_size=128,
                  train_limit=2048, valid_limit=1024, workers=0, amp=False, warmup=1),
    'local': dict(depth=16, width=2, dropout=0.3, epochs=20, batch_size=128,
                  train_limit=None, valid_limit=None, workers=0, amp=False, warmup=3),
    'kaggle': dict(depth=28, width=10, dropout=0.3, epochs=200, batch_size=128,
                   train_limit=None, valid_limit=None, workers=2, amp=True, warmup=5),
}
if RUN_MODE not in PROFILES:
    raise ValueError(f'CIFAR10_RUN_MODE 必须为 {tuple(PROFILES)}，当前为 {RUN_MODE!r}')
CFG = PROFILES[RUN_MODE].copy()
CFG['name'] = f"wrn{CFG['depth']}_{CFG['width']}"
CFG['signature'] = f"recipe-v2-{RUN_MODE}-d{CFG['depth']}-w{CFG['width']}-e{CFG['epochs']}"

IS_KAGGLE = Path('/kaggle/working').is_dir()
OUTPUT_DIR = Path(os.getenv('CIFAR10_OUTPUT_DIR', '/kaggle/working' if IS_KAGGLE else str(Path.cwd()))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LAST_PATH = OUTPUT_DIR / 'last_checkpoint.pth'
EXTENSION_PATH = OUTPUT_DIR / 'last_extension_checkpoint.pth'
BEST_PATH = OUTPUT_DIR / ('cifar10_wrn28_10_best.pth' if RUN_MODE == 'kaggle' else f"cifar10_{CFG['name']}_best.pth")

print('PyTorch:', torch.__version__)
print('设备:', device, '| 模式:', RUN_MODE, '| 配置:', CFG)
print('输出目录:', OUTPUT_DIR)

PyTorch: 2.2.2
设备: cpu | 模式: local | 配置: {'depth': 16, 'width': 2, 'dropout': 0.3, 'epochs': 20, 'batch_size': 128, 'train_limit': None, 'valid_limit': None, 'workers': 0, 'amp': False, 'warmup': 3, 'name': 'wrn16_2', 'signature': 'recipe-v2-local-d16-w2-e20'}
输出目录: /Users/bob.li/Code/DL-ZJU/final


## 1. 定位并校验数据

数据路径优先读取 `CIFAR10_DATA_DIR`；否则检查本地 `final/data` 和 Kaggle `/kaggle/input`。

In [2]:
REQUIRED_FILES = ('data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta')

def is_cifar_dir(path):
    path = Path(path)
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_FILES)

def discover_data_dir():
    env_path = os.getenv('CIFAR10_DATA_DIR')
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    cwd = Path.cwd()
    candidates.extend([cwd / 'data', cwd / 'final' / 'data', cwd.parent / 'final' / 'data'])
    for candidate in candidates:
        if is_cifar_dir(candidate):
            return candidate.resolve()
    kaggle_root = Path('/kaggle/input')
    if kaggle_root.is_dir():
        matches = sorted(p.parent for p in kaggle_root.rglob('data_batch_1') if is_cifar_dir(p.parent))
        if len(matches) == 1:
            return matches[0].resolve()
        if len(matches) > 1:
            raise RuntimeError(f'Kaggle 中找到多个 CIFAR-10 目录，请设置 CIFAR10_DATA_DIR: {matches}')
    raise FileNotFoundError('未找到 CIFAR-10 data_batch_1~5/test_batch。请设置 CIFAR10_DATA_DIR。')

def load_pickle(path):
    with Path(path).open('rb') as handle:
        return pickle.load(handle, encoding='bytes')

DATA_DIR = discover_data_dir()
meta = load_pickle(DATA_DIR / 'batches.meta')
meta_names = tuple(x.decode('utf-8') if isinstance(x, bytes) else str(x) for x in meta[b'label_names'])
assert meta_names == CLASS_NAMES, (meta_names, CLASS_NAMES)

def load_batches(names):
    xs, ys = [], []
    for name in names:
        batch = load_pickle(DATA_DIR / name)
        x = np.asarray(batch[b'data'], dtype=np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        y = np.asarray(batch[b'labels'], dtype=np.int64)
        xs.append(x); ys.append(y)
    return np.concatenate(xs), np.concatenate(ys)

train_images, train_labels = load_batches([f'data_batch_{i}' for i in range(1, 6)])
valid_images, valid_labels = load_batches(['test_batch'])
assert train_images.shape == (50000, 32, 32, 3)
assert valid_images.shape == (10000, 32, 32, 3)
assert set(np.unique(train_labels)) == set(range(10))
assert set(np.unique(valid_labels)) == set(range(10))
valid_counts = np.bincount(valid_labels, minlength=10)
assert np.all(valid_counts == 1000), valid_counts
print('数据目录:', DATA_DIR)
print('训练集:', train_images.shape, '| 验证集:', valid_images.shape)
print('验证集每类数量:', dict(zip(CLASS_NAMES, valid_counts.tolist())))

数据目录: /Users/bob.li/Code/DL-ZJU/final/data
训练集: (50000, 32, 32, 3) | 验证集: (10000, 32, 32, 3)
验证集每类数量: {'airplane': 1000, 'automobile': 1000, 'bird': 1000, 'cat': 1000, 'deer': 1000, 'dog': 1000, 'frog': 1000, 'horse': 1000, 'ship': 1000, 'truck': 1000}


In [3]:
class CIFAR10PickleDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        image = Image.fromarray(self.images[index])
        if self.transform is not None:
            image = self.transform(image)
        return image, int(self.labels[index])

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
])
test_transform = transforms.ToTensor()  # 归一化内置于 net，保证最终模型自包含

full_train_dataset = CIFAR10PickleDataset(train_images, train_labels, train_transform)
full_valid_dataset = CIFAR10PickleDataset(valid_images, valid_labels, test_transform)

def stratified_subset(labels, limit, seed):
    if limit is None or limit >= len(labels):
        return None
    rng = np.random.default_rng(seed)
    per_class = limit // 10
    remainder = limit % 10
    selected = []
    for cls in range(10):
        indices = np.flatnonzero(labels == cls)
        rng.shuffle(indices)
        selected.extend(indices[:per_class + (cls < remainder)].tolist())
    rng.shuffle(selected)
    return selected

train_indices = stratified_subset(train_labels, CFG['train_limit'], SEED)
valid_indices = stratified_subset(valid_labels, CFG['valid_limit'], SEED + 1)
train_dataset = Subset(full_train_dataset, train_indices) if train_indices is not None else full_train_dataset
valid_dataset = Subset(full_valid_dataset, valid_indices) if valid_indices is not None else full_valid_dataset

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['workers'], pin_memory=device.type == 'cuda',
                          persistent_workers=CFG['workers'] > 0, generator=loader_generator)
valid_loader = DataLoader(valid_dataset, batch_size=256, shuffle=False,
                          num_workers=CFG['workers'], pin_memory=device.type == 'cuda',
                          persistent_workers=CFG['workers'] > 0)
print('当前用于训练/验证的样本数:', len(train_dataset), len(valid_dataset))

当前用于训练/验证的样本数: 50000 10000


## 2. Wide ResNet 与自包含推理接口

`net` 内部完成 CIFAR-10 均值/方差归一化；评估模式下自动对原图和水平翻转图的 logits 取平均。

In [4]:
class WideBasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, dropout):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.shortcut = None
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False)

    def forward(self, x):
        activated = F.relu(self.bn1(x), inplace=True)
        shortcut = x if self.shortcut is None else self.shortcut(activated)
        out = self.conv1(activated)
        out = self.conv2(self.dropout(F.relu(self.bn2(out), inplace=True)))
        return out + shortcut

class WideResNetBackbone(nn.Module):
    def __init__(self, depth, width, dropout, num_classes=10):
        super().__init__()
        if (depth - 4) % 6 != 0:
            raise ValueError('Wide ResNet depth 必须满足 depth = 6n + 4')
        blocks_per_group = (depth - 4) // 6
        channels = [16, 16 * width, 32 * width, 64 * width]
        self.stem = nn.Conv2d(3, channels[0], 3, padding=1, bias=False)
        groups = []
        in_channels = channels[0]
        for out_channels, first_stride in zip(channels[1:], (1, 2, 2)):
            for block_index in range(blocks_per_group):
                stride = first_stride if block_index == 0 else 1
                groups.append(WideBasicBlock(in_channels, out_channels, stride, dropout))
                in_channels = out_channels
        self.groups = nn.Sequential(*groups)
        self.final_bn = nn.BatchNorm2d(in_channels)
        self.classifier = nn.Linear(in_channels, num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Linear):
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.groups(self.stem(x))
        x = F.relu(self.final_bn(x), inplace=True)
        return self.classifier(F.adaptive_avg_pool2d(x, 1).flatten(1))

class CIFAR10WideResNet(nn.Module):
    def __init__(self, depth, width, dropout=0.3, use_tta=True):
        super().__init__()
        self.backbone = WideResNetBackbone(depth, width, dropout, num_classes=10)
        self.use_tta = use_tta
        self.register_buffer('input_mean', torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1))
        self.register_buffer('input_std', torch.tensor((0.2470, 0.2435, 0.2616)).view(1, 3, 1, 1))

    def normalize(self, x):
        if x.dtype == torch.uint8:
            x = x.float().div(255.0)
        return (x - self.input_mean) / self.input_std

    def forward(self, x):
        normalized = self.normalize(x)
        logits = self.backbone(normalized)
        if not self.training and self.use_tta:
            flipped_logits = self.backbone(torch.flip(normalized, dims=(3,)))
            logits = (logits + flipped_logits) * 0.5
        return logits

def build_model():
    return CIFAR10WideResNet(CFG['depth'], CFG['width'], CFG['dropout'], use_tta=True)

probe = build_model()
print('模型参数量:', f'{sum(p.numel() for p in probe.parameters()):,}')
del probe

模型参数量: 691,674


## 3. CutMix、EMA、指标与断点续训

In [5]:
def smooth_one_hot(targets, num_classes=10, smoothing=0.1):
    result = torch.full((targets.size(0), num_classes), smoothing / num_classes, device=targets.device)
    return result.scatter_(1, targets[:, None], 1.0 - smoothing + smoothing / num_classes)

def cutmix_batch(images, targets, probability=0.5, alpha=1.0, smoothing=0.1):
    soft_targets = smooth_one_hot(targets, smoothing=smoothing)
    if images.size(0) < 2 or random.random() >= probability:
        return images, soft_targets
    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(images.size(0), device=images.device)
    height, width = images.shape[-2:]
    cut_ratio = math.sqrt(1.0 - lam)
    cut_w, cut_h = int(width * cut_ratio), int(height * cut_ratio)
    center_x, center_y = random.randrange(width), random.randrange(height)
    x1, x2 = max(center_x - cut_w // 2, 0), min(center_x + cut_w // 2, width)
    y1, y2 = max(center_y - cut_h // 2, 0), min(center_y + cut_h // 2, height)
    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[permutation, :, y1:y2, x1:x2]
    adjusted_lam = 1.0 - ((x2 - x1) * (y2 - y1) / (width * height))
    mixed_targets = adjusted_lam * soft_targets + (1.0 - adjusted_lam) * soft_targets[permutation]
    return mixed, mixed_targets

def soft_cross_entropy(logits, targets):
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()
        self.decay = decay
        self.updates = 0
        for parameter in self.module.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        decay = self.decay * (1.0 - math.exp(-self.updates / 2000.0))
        source = model.state_dict()
        for key, ema_value in self.module.state_dict().items():
            model_value = source[key].detach()
            if ema_value.is_floating_point():
                ema_value.mul_(decay).add_(model_value, alpha=1.0 - decay)
            else:
                ema_value.copy_(model_value)

def learning_rate(epoch, total_epochs, base_lr, warmup_epochs, min_lr=1e-5):
    if warmup_epochs and epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs - 1)
    progress = min(max(progress, 0.0), 1.0)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group['lr'] = lr

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    confusion = torch.zeros(10, 10, dtype=torch.long)
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        predictions = model(images).argmax(dim=1).cpu()
        targets = targets.long()
        confusion += torch.bincount(targets * 10 + predictions, minlength=100).reshape(10, 10)
    totals = confusion.sum(dim=1).clamp_min(1)
    class_accuracy = confusion.diag().float() / totals
    overall = confusion.diag().sum().float().item() / confusion.sum().item()
    target_macro = class_accuracy[list(TARGET_CLASSES)].mean().item()
    return dict(overall=overall, target_macro=target_macro, class_accuracy=class_accuracy.numpy(), confusion=confusion.numpy())

def atomic_torch_save(payload, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def metric_is_better(metrics, best_target, best_overall):
    return metrics['target_macro'] > best_target + 1e-12 or (
        abs(metrics['target_macro'] - best_target) <= 1e-12 and metrics['overall'] > best_overall
    )

In [6]:
def make_grad_scaler(enabled):
    amp_module = getattr(torch, 'amp', None)
    if amp_module is not None and hasattr(amp_module, 'GradScaler'):
        return amp_module.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)  # 兼容 PyTorch < 2.3

def amp_autocast(enabled):
    amp_module = getattr(torch, 'amp', None)
    if amp_module is not None and hasattr(amp_module, 'autocast'):
        return amp_module.autocast(device_type='cuda', enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)  # 兼容旧版 PyTorch

def train_one_epoch(model, ema, loader, optimizer, scaler, use_amp):
    model.train()
    running_loss, sample_count = 0.0, 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        images, soft_targets = cutmix_batch(images, targets, probability=0.5, alpha=1.0, smoothing=0.1)
        optimizer.zero_grad(set_to_none=True)
        with amp_autocast(enabled=use_amp):
            loss = soft_cross_entropy(model(images), soft_targets)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'训练损失异常: {loss.item()}')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        running_loss += loss.item() * images.size(0)
        sample_count += images.size(0)
    return running_loss / sample_count

def maybe_resume(model, ema, optimizer, scaler, checkpoint_path, signature):
    if not checkpoint_path.is_file() or os.getenv('CIFAR10_RESUME', '1') == '0':
        return 0, -1.0, -1.0, []
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if checkpoint.get('signature') != signature:
        print(f'忽略不兼容断点: {checkpoint.get("signature")} != {signature}')
        return 0, -1.0, -1.0, []
    model.load_state_dict(checkpoint['model'])
    ema.module.load_state_dict(checkpoint['ema'])
    ema.updates = int(checkpoint.get('ema_updates', 0))
    optimizer.load_state_dict(checkpoint['optimizer'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = int(checkpoint['epoch']) + 1
    print(f'从 {checkpoint_path.name} 恢复，下一个 epoch: {start_epoch + 1}')
    return start_epoch, float(checkpoint['best_target']), float(checkpoint['best_overall']), checkpoint.get('history', [])

def run_training(model, ema, optimizer, scaler, total_epochs, base_lr, warmup_epochs,
                 checkpoint_path, signature, initial_best=(-1.0, -1.0), allow_resume=True):
    if allow_resume:
        start_epoch, best_target, best_overall, history = maybe_resume(
            model, ema, optimizer, scaler, checkpoint_path, signature
        )
    else:
        start_epoch, history = 0, []
        best_target, best_overall = initial_best
    use_amp = CFG['amp'] and device.type == 'cuda'
    for epoch in range(start_epoch, total_epochs):
        started = time.perf_counter()
        lr = learning_rate(epoch, total_epochs, base_lr, warmup_epochs)
        set_optimizer_lr(optimizer, lr)
        train_loss = train_one_epoch(model, ema, train_loader, optimizer, scaler, use_amp)
        metrics = evaluate(ema.module, valid_loader)
        record = dict(epoch=epoch + 1, lr=lr, train_loss=train_loss,
                      overall=metrics['overall'], target_macro=metrics['target_macro'])
        history.append(record)
        if metric_is_better(metrics, best_target, best_overall):
            best_target, best_overall = metrics['target_macro'], metrics['overall']
            atomic_torch_save(ema.module.state_dict(), BEST_PATH)
        checkpoint = dict(signature=signature, epoch=epoch, model=model.state_dict(),
                          ema=ema.module.state_dict(), ema_updates=ema.updates, optimizer=optimizer.state_dict(),
                          scaler=scaler.state_dict(), best_target=best_target,
                          best_overall=best_overall, history=history)
        atomic_torch_save(checkpoint, checkpoint_path)
        elapsed = time.perf_counter() - started
        print(f"Epoch {epoch + 1:03d}/{total_epochs} | {elapsed/60:.1f} min | lr={lr:.5f} | "
              f"loss={train_loss:.4f} | acc={metrics['overall']:.2%} | target={metrics['target_macro']:.2%}")
    return best_target, best_overall, history

## 4. 训练

程序每个 epoch 写入 `last_checkpoint.pth`；重启 Notebook 时会自动续训。

In [7]:
net = build_model().to(device)
ema = ModelEMA(net, decay=0.999)
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
scaler = make_grad_scaler(enabled=CFG['amp'] and device.type == 'cuda')

best_target, best_overall, history = run_training(
    net, ema, optimizer, scaler, total_epochs=CFG['epochs'], base_lr=0.1,
    warmup_epochs=CFG['warmup'], checkpoint_path=LAST_PATH, signature=CFG['signature']
)

if RUN_MODE == 'kaggle' and best_target < 0.95:
    print(f'目标三类最佳平均 {best_target:.2%} < 95%，从最佳 EMA 权重追加 50 epochs。')
    net.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
    ema = ModelEMA(net, decay=0.999)
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scaler = make_grad_scaler(enabled=True)
    extension_signature = CFG['signature'] + '-extension50'
    best_target, best_overall, extension_history = run_training(
        net, ema, optimizer, scaler, total_epochs=50, base_lr=0.01, warmup_epochs=0,
        checkpoint_path=EXTENSION_PATH, signature=extension_signature,
        initial_best=(best_target, best_overall), allow_resume=True
    )
    history.extend(extension_history)

print(f'最佳总体准确率: {best_overall:.2%}')
print(f'最佳飞机/猫/青蛙平均准确率: {best_target:.2%}')
print('最佳权重:', BEST_PATH)

从 last_checkpoint.pth 恢复，下一个 epoch: 21
最佳总体准确率: 87.15%
最佳飞机/猫/青蛙平均准确率: 84.33%
最佳权重: /Users/bob.li/Code/DL-ZJU/final/cifar10_wrn16_2_best.pth


## 5. 加载最佳权重并输出分类指标

In [8]:
if not BEST_PATH.is_file():
    raise FileNotFoundError(f'最佳权重不存在: {BEST_PATH}')
net = build_model().to(device)
net.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
net.eval()
final_metrics = evaluate(net, valid_loader)

for index, accuracy in enumerate(final_metrics['class_accuracy']):
    marker = '  <== 评分类' if index in TARGET_CLASSES else ''
    print(f'{index}: {CLASS_NAMES[index]:10s} {accuracy:.2%}{marker}')
print()
print(f"总体准确率: {final_metrics['overall']:.2%}")
print(f"飞机/猫/青蛙平均准确率: {final_metrics['target_macro']:.2%}")

try:
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(final_metrics['confusion'], cmap='Blues')
    ax.set(xticks=range(10), yticks=range(10), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
           xlabel='Predicted label', ylabel='True label', title='CIFAR-10 confusion matrix')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    plt.show()
except Exception as exc:
    print('当前环境无法绘制混淆矩阵，数值矩阵如下:', type(exc).__name__, exc)
    print(final_metrics['confusion'])

0: airplane   87.80%  <== 评分类
1: automobile 96.40%
2: bird       82.10%
3: cat        71.70%  <== 评分类
4: deer       86.60%
5: dog        79.50%
6: frog       93.50%  <== 评分类
7: horse      88.90%
8: ship       94.40%
9: truck      90.60%

总体准确率: 87.15%
飞机/猫/青蛙平均准确率: 84.33%
当前环境无法绘制混淆矩阵，数值矩阵如下: ImportError cannot import name 'backend2gui' from 'IPython.core.pylabtools' (/opt/anaconda3/envs/notebook/lib/python3.11/site-packages/IPython/core/pylabtools.py)
[[878   5  43   7   6   1   8   5  36  11]
 [  2 964   1   2   0   0   2   0   4  25]
 [ 36   0 821  26  40  19  50   4   4   0]
 [ 10   2  49 717  39 106  53  15   6   3]
 [  3   1  30  26 866   7  42  24   1   0]
 [  1   1  25 106  33 795  17  22   0   0]
 [  5   1  22  18   6  12 935   0   1   0]
 [  8   0  15  21  29  34   2 889   0   2]
 [ 20  10   9   4   0   1   4   1 944   7]
 [ 13  51   3   9   1   0   2   2  13 906]]


## 6. 助教统一测试接口

以下单元格严格按课程设计 PDF 第 6 页执行：使用变量 `net` 保存纯 `state_dict`，重新创建 `net`、加载权重并调用 `net.eval()`。

In [9]:
# 按课程设计 PDF 第 6 页要求，保存和重新加载时的模型变量统一命名为 net。
FINAL_MODEL_PATH = OUTPUT_DIR / f"cifar10_{CFG['name']}.pth"

# 先把按目标三类平均准确率选出的最佳 EMA 权重装入 net，再保存纯 state_dict。
net = build_model().to(device)
net.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
net.eval()
torch.save(net.state_dict(), FINAL_MODEL_PATH)

# 记录保存前的真实 batch 输出。
verification_images, _ = next(iter(valid_loader))
verification_images = verification_images[:16].to(device)
with torch.no_grad():
    expected_logits = net(verification_images)

# 重新创建 net，加载刚保存的权重，并进入评估模式。
net = build_model().to(device)
net.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=device, weights_only=True))
net.eval()
with torch.no_grad():
    reloaded_logits = net(verification_images)
torch.testing.assert_close(expected_logits, reloaded_logits, rtol=0, atol=0)
assert not net.training
assert expected_logits.shape == (16, 10)
print(f'最终验证通过: {FINAL_MODEL_PATH.name} 已由 net 保存并重新加载，net.eval() 已生效，预测完全一致。')

最终验证通过: cifar10_wrn16_2.pth 已由 net 保存并重新加载，net.eval() 已生效，预测完全一致。


## Kaggle 手动运行清单（本 Notebook 不执行上传或提交）

1. Notebook 名称设为 `3250102780_李昊泽`。
2. 附加课程 CIFAR-10 数据并打开 GPU；CUDA 环境会自动使用 `kaggle` 配置。
3. 手动运行全部单元格。
4. 确认最后一个代码单元格输出“最终验证通过”，且供助教统一测试的 `/kaggle/working/cifar10_wrn28_10.pth` 存在；`*_best.pth` 仅作为训练中的最佳检查点。